# Pipeline Hán–Việt: Ensemble Alignment → Qwen Verification → Re-Alignment → Fine-tune MarianMT

Notebook này thực hiện toàn bộ quy trình tự động từ **dóng hàng câu song ngữ (Hán–Việt)** đến **huấn luyện mô hình dịch máy** trên Kaggle GPU, gồm 5 giai đoạn kỹ thuật chính:

```
Raw CSV (Hán + Việt)
       │
       ▼
  [Phase 1] Ensemble Alignment
            LaBSE + Vecalign + BERTAlign + SimAlign → DP Alignment
       │
       ▼
  [Phase 2] Qwen2.5-7B Verification (Offline, 4-bit)
            Lọc các cặp câu trong vùng bất định [0.32, 0.50]
       │
       ▼
  [Phase 3] Gemini API Re-Alignment
            Dóng hàng lại các cụm NaN còn sót
       │
       ▼
  [Phase 4] prepare_data.py
            Gộp TSV → train.json / val.json (90/10 split)
       │
       ▼
  [Phase 5] Fine-tune Helsinki-NLP/opus-mt-zh-vi
            MarianMT, 20 epochs, lr=5e-5, warmup=10%
```


## Bước 1: Clone / Cập nhật Repository

In [ ]:
GITHUB_REPO_URL = "https://github.com/quachthanhhmd/SinoNom-NLP.git"

import os
repo_name = GITHUB_REPO_URL.split("/")[-1].replace(".git", "")

if not os.path.exists(repo_name):
    print(f"Cloning {GITHUB_REPO_URL} ...")
    !git clone -b features/mapping-translation {GITHUB_REPO_URL}
else:
    print("Repo đã tồn tại — đang cập nhật lên commit mới nhất...")
    %cd {repo_name}
    !git fetch --all
    !git reset --hard origin/features/mapping-translation
    %cd ..

%cd {repo_name}
!ls -la

## Bước 2: Cài đặt Thư viện

In [ ]:
!pip install -q sentence-transformers pandas openpyxl setuptools
!pip install -q transformers[torch] datasets evaluate sacrebleu accelerate tensorboard
!pip install -q bitsandbytes
!pip install -q git+https://github.com/cisnlp/simalign.git

## Bước 2.5: Nạp Gemini API Key (cần cho Phase 3)

Truy cập **Add-ons → Secrets** ở thanh bên phải Kaggle, thêm secret với Label = `GEMINI_API_KEY`  
và Value = danh sách key cách nhau bằng dấu phẩy (ví dụ: `key1,key2,key3`). Sau đó chạy cell dưới:

In [ ]:
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["GEMINI_API_KEY"] = UserSecretsClient().get_secret("GEMINI_API_KEY")
    keys = os.environ["GEMINI_API_KEY"].split(",")
    print(f"✅ Đã nạp thành công {len(keys)} Gemini API Key(s).")
except Exception as e:
    print(f"⚠️  Không tìm thấy Kaggle Secret: {e}")
    print("Nếu bạn chạy local, hãy tự set: os.environ['GEMINI_API_KEY'] = 'your_key'")

## Bước 3: Phân Tích Dữ Liệu Thô (EDA trước khi dóng hàng)

Thống kê số câu Hán / Việt theo từng nhóm quyển và phân phối độ dài câu.

In [ ]:
import os, re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from IPython.display import display

SINO_DIR = 'dataset/MAPPING/sino_extract'
VIET_DIR = 'dataset/MAPPING/vietnam_extract/csv'

MAPPING_GROUPS = [
    ('Q01',     ['q1_sentences.csv'],                              'q01.csv'),
    ('Q02-04',  ['q2_sentences.csv','q3_sentences.csv','q4_sentences.csv'], 'q2_3_4.csv'),
    ('Q05',     ['q5_sentences.csv'],                              'q05.csv'),
    ('Q06',     ['q6_sentences.csv'],                              'q6.csv'),
    ('Q07-08',  ['q7_sentences.csv','q8_sentences.csv'],           'q07_08.csv'),
    ('Q09',     ['q9_sentences.csv'],                              'q09.csv'),
    ('Q10-11',  ['q10_11_sentences.csv'],                          'q10_11.csv'),
    ('Q12',     ['q12_sentences.csv'],                             'q12.csv'),
    ('Q13',     ['q13_sentences.csv'],                             'q13.csv'),
    ('Q14-15',  ['q14_sentences.csv','q15_sentences.csv'],         'q14_15.csv'),
    ('Q16-17',  ['q16_17_sentences.csv'],                          'q16_17.csv'),
]

def detect_sep(fp):
    with open(fp, 'r', encoding='utf-8') as f:
        return ';' if ';' in f.readline() else ','

def load_sino(fp):
    sep = detect_sep(fp)
    rows = []
    with open(fp, 'r', encoding='utf-8') as f:
        f.readline()
        for line in f:
            line = line.strip()
            if not line: continue
            parts = line.split(sep, 1)
            if len(parts) < 2: continue
            rest = parts[1]
            for pat in [sep+'"[', sep+'[', sep+'[]']:
                idx = rest.rfind(pat)
                if idx > -1:
                    rest = rest[:idx]; break
            else:
                idx = rest.rfind(sep)
                if idx > -1: rest = rest[:idx]
            rest = rest.strip().strip('"')
            rows.append(rest)
    return rows

records = []
all_han_lens, all_viet_lens = [], []

for group, sino_files, viet_file in MAPPING_GROUPS:
    han_sents = []
    for sf in sino_files:
        p = os.path.join(SINO_DIR, sf)
        if os.path.exists(p):
            sents = load_sino(p)
            han_sents.extend(sents)
            all_han_lens.extend(len(s) for s in sents)
    vp = os.path.join(VIET_DIR, viet_file)
    viet_count = 0
    if os.path.exists(vp):
        dfv = pd.read_csv(vp)
        viet_count = len(dfv)
        all_viet_lens.extend(dfv['sentence'].dropna().apply(lambda x: len(str(x).split())).tolist())
    ratio = len(han_sents)/viet_count if viet_count else 0
    records.append({'Nhóm': group, 'Câu Hán': len(han_sents),
                    'Câu Việt (raw)': viet_count, 'Tỷ lệ Hán:Việt': round(ratio,2)})

df_eda = pd.DataFrame(records)
df_eda.loc[len(df_eda)] = ['TỔNG', df_eda['Câu Hán'].sum(),
                            df_eda['Câu Việt (raw)'].sum(), '']
print('=== THỐNG KÊ DỮ LIỆU THÔ ===')
display(df_eda.style.set_caption('Bảng 1. Thống kê số lượng câu Hán và Việt theo nhóm quyển'))


In [ ]:
# Biểu đồ 1: Phân phối độ dài câu
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Phân phối độ dài câu trong Dataset thô', fontsize=14, fontweight='bold')

axes[0].hist(np.clip(all_han_lens, 0, 200), bins=40, color='#4C72B0', edgecolor='white', alpha=0.85)
axes[0].set_title('Câu Hán (số ký tự)')
axes[0].set_xlabel('Số ký tự'); axes[0].set_ylabel('Số câu')
axes[0].axvline(np.median(all_han_lens), color='red', linestyle='--', label=f'Median = {np.median(all_han_lens):.0f}')
axes[0].legend()

axes[1].hist(np.clip(all_viet_lens, 0, 120), bins=40, color='#DD8452', edgecolor='white', alpha=0.85)
axes[1].set_title('Câu Việt (số từ)')
axes[1].set_xlabel('Số từ'); axes[1].set_ylabel('Số câu')
axes[1].axvline(np.median(all_viet_lens), color='red', linestyle='--', label=f'Median = {np.median(all_viet_lens):.0f}')
axes[1].legend()

plt.tight_layout()
plt.savefig('fig_01_raw_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Hình 1. Phân phối độ dài câu Hán và Việt trước khi dóng hàng.')

## Bước 4: Phase 1 — Ensemble Alignment (Full Dataset)

Chạy bộ dóng hàng kết hợp gồm 4 scorer: **LaBSE**, **Vecalign**, **BERTAlign (MiniLM-L12)**, **SimAlign (XLM-R)**.  
Kết quả ma trận điểm số tổng hợp (fused) được đưa qua thuật toán **Quy hoạch động (DP)** để tìm tổ hợp dóng hàng tối ưu toàn cục.  
Ngưỡng DP: `threshold=0.32`, `skip_penalty=0.05`, `max_merge_han=15`, `max_merge_viet=2`.

In [ ]:
!python run_mapping.py \
    --aligner ensemble \
    --sino_dir dataset/MAPPING/sino_extract \
    --viet_dir dataset/MAPPING/vietnam_extract/csv \
    --output_dir output \
    --work_code HVB_001 \
    --device cuda

### Kết quả Phase 1 — Thống kê tổng hợp theo từng Volume

In [ ]:
import glob
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display

tsv_files = sorted(glob.glob('output/HVB_001/**/*.tsv', recursive=True))
print(f'Tìm thấy {len(tsv_files)} file TSV kết quả Phase 1.')

p1_records = []
for fp in tsv_files:
    vol = os.path.basename(os.path.dirname(fp))
    df = pd.read_csv(fp, sep='\t')
    total = len(df)
    matched = df['han_sentence'].notna() & df['viet_sentence'].notna()
    nan_count = (~matched).sum()
    p1_records.append({
        'Volume': vol,
        'Tổng dòng': total,
        'Matched': matched.sum(),
        'NaN (Unmatched)': nan_count,
        'Match Rate (%)': round(matched.mean()*100, 1)
    })

df_p1 = pd.DataFrame(p1_records)
df_p1.loc[len(df_p1)] = ['TỔNG / TB', df_p1['Tổng dòng'].sum(), df_p1['Matched'].sum(),
                          df_p1['NaN (Unmatched)'].sum(), round(df_p1['Match Rate (%)'].mean(), 1)]
print('\n=== KẾT QUẢ PHASE 1 — ENSEMBLE ALIGNMENT ===')
display(df_p1.style.set_caption('Bảng 2. Thống kê kết quả dóng hàng Phase 1 theo từng Volume'))


In [ ]:
# Biểu đồ 2: Matched vs Unmatched Phase 1
df_plot = df_p1[df_p1['Volume'] != 'TỔNG / TB'].copy()
x = np.arange(len(df_plot))
w = 0.35

fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(x - w/2, df_plot['Matched'], w, label='Matched', color='#55A868', alpha=0.9)
ax.bar(x + w/2, df_plot['NaN (Unmatched)'], w, label='Unmatched (NaN)', color='#C44E52', alpha=0.9)
ax.set_xticks(x); ax.set_xticklabels(df_plot['Volume'], rotation=30, ha='right')
ax.set_ylabel('Số dòng')
ax.set_title('Hình 2. Kết quả dóng hàng Phase 1 theo Volume', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('fig_02_phase1_results.png', dpi=150, bbox_inches='tight')
plt.show()

## Bước 5: Phase 2 — Qwen2.5-7B Verification (Full Dataset, Offline 4-bit)

Qwen2.5-7B-Instruct nạp ở chế độ **4-bit quantization** (bitsandbytes) để vừa với VRAM T4 (15GB).  
Chỉ các cặp câu có điểm ensemble nằm trong **vùng bất định [0.32, 0.50]** mới được gửi lên Qwen để chấm điểm 0–5.  
Các cặp bị chấm điểm thấp (< 4) sẽ bị **tách ra thành 2 dòng NaN** để Phase 3 xử lý lại.

In [ ]:
!python run_mapping.py \
    --aligner ensemble \
    --qwen \
    --sino_dir dataset/MAPPING/sino_extract \
    --viet_dir dataset/MAPPING/vietnam_extract/csv \
    --output_dir output \
    --work_code HVB_001 \
    --device cuda

### Kết quả Phase 2 — So sánh NaN trước và sau xác thực

In [ ]:
p2_records = []
for fp in sorted(glob.glob('output/HVB_001/**/*.tsv', recursive=True)):
    vol = os.path.basename(os.path.dirname(fp))
    df = pd.read_csv(fp, sep='\t')
    total = len(df)
    matched = df['han_sentence'].notna() & df['viet_sentence'].notna()
    nan_count = (~matched).sum()
    # Lấy thông tin điểm qwen từ cột nếu có
    has_qwen = 'qwen_score' in df.columns
    p2_records.append({
        'Volume': vol,
        'Tổng dòng P2': total,
        'Matched P2': matched.sum(),
        'NaN P2': nan_count,
        'Match Rate P2 (%)': round(matched.mean()*100, 1),
        'Có điểm Qwen': 'Có' if has_qwen else 'Không'
    })

df_p2 = pd.DataFrame(p2_records)
df_p2.loc[len(df_p2)] = ['TỔNG / TB', df_p2['Tổng dòng P2'].sum(), df_p2['Matched P2'].sum(),
                          df_p2['NaN P2'].sum(), round(df_p2['Match Rate P2 (%)'].mean(),1), '']
print('\n=== KẾT QUẢ PHASE 2 — QWEN VERIFICATION ===')
display(df_p2.style.set_caption('Bảng 3. Kết quả sau Phase 2 — Qwen LLM Verification'))


In [ ]:
# Biểu đồ 3: Phân phối điểm qwen_score trên toàn dataset
all_scores = []
for fp in sorted(glob.glob('output/HVB_001/**/*.tsv', recursive=True)):
    df = pd.read_csv(fp, sep='\t')
    if 'qwen_score' in df.columns:
        scores = df['qwen_score'].dropna()
        all_scores.extend(scores.tolist())

if all_scores:
    fig, ax = plt.subplots(figsize=(9, 5))
    score_counts = pd.Series(all_scores).value_counts().sort_index()
    bars = ax.bar(score_counts.index.astype(int), score_counts.values,
                  color=['#C44E52','#E07A5F','#F2CC8F','#81B29A','#3D405B','#2D6A4F'],
                  edgecolor='white', alpha=0.9)
    ax.bar_label(bars, fmt='%d')
    ax.set_xlabel('Điểm Qwen (0 = sai hoàn toàn, 5 = chuẩn xác)'); ax.set_ylabel('Số cặp câu')
    ax.set_title('Hình 3. Phân phối điểm Qwen verification trên toàn bộ dataset', fontweight='bold')
    ax.set_xticks([0,1,2,3,4,5])
    plt.tight_layout()
    plt.savefig('fig_03_qwen_score_dist.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('Không có cột qwen_score trong output (Phase 2 chưa chạy hoặc chạy bằng offline Qwen).')

## Bước 6: Phase 3 — Gemini API Re-Alignment (Full Dataset)

Gemini 3.1 Flash Lite phát hiện các **NaN cluster** (cụm câu liên tiếp không có cặp đôi) và thực hiện dóng hàng lại cục bộ.  
Giới hạn: tối đa **30 câu/cụm** mỗi lần gọi API để đảm bảo độ chính xác.  
Checkpoint tự động lưu sau mỗi cluster → có thể resume nếu bị ngắt.

In [ ]:
!python run_mapping.py \
    --aligner ensemble \
    --qwen \
    --realign \
    --sino_dir dataset/MAPPING/sino_extract \
    --viet_dir dataset/MAPPING/vietnam_extract/csv \
    --output_dir output \
    --work_code HVB_001 \
    --device cuda

### Kết quả Phase 3 — Tổng kết toàn bộ Pipeline dóng hàng

In [ ]:
p3_records = []
for fp in sorted(glob.glob('output/HVB_001/**/*.tsv', recursive=True)):
    vol = os.path.basename(os.path.dirname(fp))
    df = pd.read_csv(fp, sep='\t')
    total = len(df)
    matched = df['han_sentence'].notna() & df['viet_sentence'].notna()
    nan_count = (~matched).sum()
    p3_records.append({
        'Volume': vol,
        'Tổng dòng P3': total,
        'Matched (cặp sạch)': matched.sum(),
        'NaN còn lại': nan_count,
        'Match Rate (%)': round(matched.mean()*100, 1)
    })

df_p3 = pd.DataFrame(p3_records)
total_matched = df_p3['Matched (cặp sạch)'].sum()
total_nan = df_p3['NaN còn lại'].sum()
df_p3.loc[len(df_p3)] = ['TỔNG / TB', df_p3['Tổng dòng P3'].sum(),
                          total_matched, total_nan, round(df_p3['Match Rate (%)'].mean(),1)]
print('\n=== KẾT QUẢ CUỐI CÙNG — PHASE 3 RE-ALIGNMENT ===')
display(df_p3.style.set_caption('Bảng 4. Kết quả dóng hàng cuối cùng sau Phase 3'))
print(f'\n>>> Tổng số cặp câu song ngữ sạch đưa vào huấn luyện: {total_matched:,} cặp')
print(f'>>> Số dòng NaN không thể ghép: {total_nan:,} dòng (sẽ bị loại bỏ khi prepare_data)')


In [ ]:
# Biểu đồ 4: So sánh Match Rate qua 3 Phase (nếu có dữ liệu cả 3 Phase)
# Dùng số liệu từ các bảng đã tính ở trên (df_p1, df_p2, df_p3)
try:
    vols = df_p1[df_p1['Volume'] != 'TỔNG / TB']['Volume'].tolist()
    r1 = df_p1[df_p1['Volume'] != 'TỔNG / TB']['Match Rate (%)'].tolist()
    r2 = df_p2[df_p2['Volume'] != 'TỔNG / TB']['Match Rate P2 (%)'].tolist()
    r3 = df_p3[df_p3['Volume'] != 'TỔNG / TB']['Match Rate (%)'].tolist()
    min_len = min(len(vols), len(r1), len(r2), len(r3))
    vols, r1, r2, r3 = vols[:min_len], r1[:min_len], r2[:min_len], r3[:min_len]

    x = np.arange(min_len); w = 0.25
    fig, ax = plt.subplots(figsize=(14, 5))
    ax.bar(x - w, r1, w, label='Phase 1 (Ensemble DP)', color='#4C72B0', alpha=0.85)
    ax.bar(x,     r2, w, label='Phase 2 (+ Qwen Verify)', color='#DD8452', alpha=0.85)
    ax.bar(x + w, r3, w, label='Phase 3 (+ Gemini Realign)', color='#55A868', alpha=0.85)
    ax.set_xticks(x); ax.set_xticklabels(vols, rotation=30, ha='right')
    ax.set_ylabel('Match Rate (%)')
    ax.set_title('Hình 4. So sánh Match Rate qua 3 Phase dóng hàng', fontweight='bold')
    ax.set_ylim(0, 110)
    ax.axhline(y=80, color='gray', linestyle='--', alpha=0.5, label='Ngưỡng 80%')
    ax.legend()
    plt.tight_layout()
    plt.savefig('fig_04_phase_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()
except Exception as e:
    print(f'Không thể vẽ biểu đồ so sánh (cần chạy đủ cả 3 phase): {e}')

## Bước 7: Chuẩn bị Dataset Huấn luyện (prepare_data)

Script `prepare_data.py` quét toàn bộ file TSV, lọc các dòng NaN, gộp thành một dataset duy nhất,  
sau đó chia tập **Train (90%)** và **Validation (10%)** theo tỷ lệ ngẫu nhiên có seed cố định.

In [ ]:
!python scripts/prepare_data.py
!ls -lh output/translation_dataset/

In [ ]:
# Thống kê dataset sau khi chuẩn bị
import json

def load_jsonl(path):
    records = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records

train_data = load_jsonl('output/translation_dataset/train.json')
val_data   = load_jsonl('output/translation_dataset/val.json')

def avg_len_han(data):
    return np.mean([len(d['translation']['zh']) for d in data])

def avg_len_viet(data):
    return np.mean([len(d['translation']['vi'].split()) for d in data])

df_ds = pd.DataFrame([
    {'Tập': 'Train', 'Số cặp câu': len(train_data),
     'Avg độ dài Hán (ký tự)': round(avg_len_han(train_data), 1),
     'Avg độ dài Việt (từ)': round(avg_len_viet(train_data), 1)},
    {'Tập': 'Validation', 'Số cặp câu': len(val_data),
     'Avg độ dài Hán (ký tự)': round(avg_len_han(val_data), 1),
     'Avg độ dài Việt (từ)': round(avg_len_viet(val_data), 1)},
    {'Tập': 'TỔNG', 'Số cặp câu': len(train_data)+len(val_data),
     'Avg độ dài Hán (ký tự)': '-', 'Avg độ dài Việt (từ)': '-'},
])
print('\n=== THỐNG KÊ DATASET HUẤN LUYỆN ===')
display(df_ds.style.set_caption('Bảng 5. Thông tin tập dữ liệu huấn luyện sau prepare_data'))


In [ ]:
# Biểu đồ 5: Scatter plot độ dài Hán vs Việt (kiểm tra tương quan)
han_lens = [len(d['translation']['zh']) for d in train_data]
viet_lens = [len(d['translation']['vi'].split()) for d in train_data]

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(han_lens, viet_lens, alpha=0.3, s=12, color='#4C72B0')
ax.set_xlabel('Độ dài câu Hán (số ký tự)')
ax.set_ylabel('Độ dài câu Việt (số từ)')
ax.set_title('Hình 5. Tương quan độ dài câu Hán–Việt trong tập Train', fontweight='bold')

# Đường hồi quy tuyến tính
m, b = np.polyfit(han_lens, viet_lens, 1)
x_line = np.linspace(min(han_lens), max(han_lens), 100)
ax.plot(x_line, m*x_line+b, 'r--', linewidth=1.5,
        label=f'Hồi quy tuyến tính (y={m:.2f}x+{b:.1f})')
ax.legend()
plt.tight_layout()
plt.savefig('fig_05_length_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

## Bước 8: Fine-tune Mô hình Dịch Máy (MarianMT)

Tinh chỉnh mô hình **Helsinki-NLP/opus-mt-zh-vi** (MarianMT, ~300M tham số) trên tập dữ liệu Hán–Việt cổ.  
Cấu hình huấn luyện tối ưu:
- **Epochs:** 20 | **Learning rate:** 5e-5 | **Warmup:** 10% số bước đầu | **Weight decay:** 0.01
- **Batch size:** 16/device × 2 GPU = 32 effective | **Metric đánh giá:** SacreBLEU

In [ ]:
!wget -q https://raw.githubusercontent.com/huggingface/transformers/main/examples/pytorch/translation/run_translation.py

!python run_translation.py \
    --model_name_or_path Helsinki-NLP/opus-mt-zh-vi \
    --source_lang zh \
    --target_lang vi \
    --max_source_length 512 \
    --train_file output/translation_dataset/train.json \
    --validation_file output/translation_dataset/val.json \
    --output_dir ./han_viet_translation_model \
    --per_device_train_batch_size 16 \
    --per_device_eval_batch_size 16 \
    --do_train \
    --do_eval \
    --num_train_epochs 20 \
    --learning_rate 5e-5 \
    --warmup_ratio 0.1 \
    --save_total_limit 3 \
    --weight_decay 0.01 \
    --predict_with_generate \
    --eval_strategy epoch \
    --save_strategy epoch

### Biểu đồ quá trình huấn luyện — Loss & BLEU theo Epoch

Đọc log từ `trainer_state.json` để vẽ biểu đồ Loss và BLEU qua từng epoch.

In [ ]:
import json, glob
import matplotlib.pyplot as plt

state_file = 'han_viet_translation_model/trainer_state.json'
if not os.path.exists(state_file):
    # Thử tìm trong checkpoint cuối cùng
    ckpts = sorted(glob.glob('han_viet_translation_model/checkpoint-*/trainer_state.json'))
    state_file = ckpts[-1] if ckpts else None

if state_file and os.path.exists(state_file):
    with open(state_file) as f:
        state = json.load(f)

    log_history = state.get('log_history', [])
    eval_logs = [l for l in log_history if 'eval_loss' in l]
    train_logs = [l for l in log_history if 'loss' in l and 'eval_loss' not in l]

    epochs_eval = [l['epoch'] for l in eval_logs]
    eval_loss   = [l['eval_loss'] for l in eval_logs]
    eval_bleu   = [l.get('eval_bleu', None) for l in eval_logs]

    epochs_train = [l['epoch'] for l in train_logs]
    train_loss   = [l['loss'] for l in train_logs]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle('Hình 6. Quá trình Fine-tuning MarianMT (Helsinki-NLP/opus-mt-zh-vi)', fontweight='bold')

    # Loss
    axes[0].plot(epochs_train, train_loss, label='Train Loss', color='#4C72B0', marker='o', markersize=3)
    axes[0].plot(epochs_eval, eval_loss, label='Val Loss', color='#C44E52', marker='s', linewidth=2)
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Cross-Entropy Loss')
    axes[0].set_title('Loss theo Epoch'); axes[0].legend()
    axes[0].grid(alpha=0.3)

    # BLEU
    if any(b is not None for b in eval_bleu):
        bleu_vals = [b for b in eval_bleu if b is not None]
        bleu_epochs = [e for e, b in zip(epochs_eval, eval_bleu) if b is not None]
        axes[1].plot(bleu_epochs, bleu_vals, label='Val BLEU', color='#55A868', marker='D', linewidth=2)
        axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('SacreBLEU Score')
        axes[1].set_title('BLEU Score theo Epoch'); axes[1].legend()
        axes[1].grid(alpha=0.3)
        best_epoch = bleu_epochs[bleu_vals.index(max(bleu_vals))]
        print(f'Best BLEU = {max(bleu_vals):.4f} tại Epoch {best_epoch}')

    plt.tight_layout()
    plt.savefig('fig_06_training_curves.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('Chưa tìm thấy trainer_state.json — hãy chạy bước huấn luyện (Bước 8) trước.')

## Bước 9: Kiểm thử và Đánh giá Mô hình

### 9.1 So sánh mô hình gốc vs mô hình sau fine-tune

Dịch cùng một bộ câu mẫu bằng cả hai mô hình để đánh giá mức độ cải thiện rõ ràng.

In [ ]:
import os
import torch
from transformers import MarianMTModel, MarianTokenizer

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Thiết bị sử dụng: {device}')

# Nạp mô hình gốc (chưa fine-tune)
print('\nNạp mô hình GỐC (Helsinki-NLP/opus-mt-zh-vi)...')
tok_orig = MarianTokenizer.from_pretrained('Helsinki-NLP/opus-mt-zh-vi')
mdl_orig = MarianMTModel.from_pretrained('Helsinki-NLP/opus-mt-zh-vi').to(device)

# Nạp mô hình sau fine-tune
model_path = os.path.abspath('./han_viet_translation_model')
print(f'Nạp mô hình SAU FINE-TUNE từ: {model_path}')
tok_ft  = MarianTokenizer.from_pretrained(model_path)
mdl_ft  = MarianMTModel.from_pretrained(model_path).to(device)
print('✅ Nạp thành công cả hai mô hình.')

def translate(model, tokenizer, text):
    inputs = tokenizer(text, return_tensors='pt', padding=True, truncation=True, max_length=512).to(device)
    with torch.no_grad():
        outputs = model.generate(**inputs, num_beams=6, max_length=512)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
# Bộ câu kiểm thử đa dạng (ngắn, dài, địa danh, số liệu, chức quan)
test_sentences = [
    '大南一統志卷之二承天府上',
    '嗣德二年以欽文殿爲經筵之所成，',
    '在京城外之春祿邑，',
    '明命七年建正堂前堂各三間，合為一座，正中祀風伯之神，左雲師，右雷師。',
    '紹治六年新建旗柱，通長七丈六尺五寸，上設望斗，凡朝賀巡幸以此。',
    '城池周二千四百八十七丈三尺六寸，高一丈五尺六寸，厚五丈，甃磚。',
    '又建一太醫所在，京城內東福坊，院判及醫生居焉。',
    '嘉隆三年建成，正楹十三間，前楹十五間。',
    '自安南建長月國，在陳為順化地，黎為順化承宣，均稱重鎮。',
    '廟四圍繚以飄墻，前為門樓，樓前為坊門。',
]

print('=== BENCHMARK DỊCH THỬ: MÔ HÌNH GỐC vs SAU FINE-TUNE ===\n')
rows = []
for i, sent in enumerate(test_sentences, 1):
    pred_orig = translate(mdl_orig, tok_orig, sent)
    pred_ft   = translate(mdl_ft,   tok_ft,   sent)
    rows.append({'#': i, 'Câu Hán': sent, 'Mô hình gốc': pred_orig, 'Sau fine-tune': pred_ft})

df_benchmark = pd.DataFrame(rows)
pd.set_option('display.max_colwidth', 120)
display(df_benchmark.style.set_caption('Bảng 6. Benchmark dịch thử — So sánh mô hình gốc và sau fine-tune'))

### 9.2 Đánh giá BLEU trên tập Validation

In [ ]:
from evaluate import load as eval_load
import json

bleu_metric = eval_load('sacrebleu')

def evaluate_bleu(model, tokenizer, data, desc=''):
    preds, refs = [], []
    model.eval()
    for item in data:
        src = item['translation']['zh']
        tgt = item['translation']['vi']
        pred = translate(model, tokenizer, src)
        preds.append(pred)
        refs.append([tgt])
    score = bleu_metric.compute(predictions=preds, references=refs)
    print(f'{desc}: SacreBLEU = {score["score"]:.4f}')
    return score['score'], preds, refs

print('Đang tính BLEU trên tập Validation...')
bleu_orig, preds_orig, refs_val = evaluate_bleu(mdl_orig, tok_orig, val_data, 'Mô hình GỐC')
bleu_ft,   preds_ft,   _        = evaluate_bleu(mdl_ft,   tok_ft,   val_data, 'Sau FINE-TUNE')

improvement = bleu_ft - bleu_orig
print(f'\n>>> Cải thiện BLEU sau fine-tune: +{improvement:.4f} điểm')

df_bleu = pd.DataFrame([
    {'Mô hình': 'Helsinki-NLP/opus-mt-zh-vi (Gốc)', 'SacreBLEU': round(bleu_orig, 4)},
    {'Mô hình': 'opus-mt-zh-vi + Fine-tune (Hán-Việt cổ)', 'SacreBLEU': round(bleu_ft, 4)},
])
display(df_bleu.style.set_caption('Bảng 7. So sánh SacreBLEU trên tập Validation'))

### 9.3 Phân tích lỗi (Error Analysis) — 5 câu có BLEU thấp nhất

In [ ]:
from sacrebleu.metrics import BLEU as SacreBLEUMetric

# Tính BLEU từng câu
sent_bleus = []
bleu_sent_metric = SacreBLEUMetric(effective_order=True)

for pred, ref_list in zip(preds_ft, refs_val):
    score = bleu_sent_metric.sentence_score(pred, ref_list).score
    sent_bleus.append(score)

# Lấy 5 câu có điểm thấp nhất
idx_worst = sorted(range(len(sent_bleus)), key=lambda i: sent_bleus[i])[:5]

error_rows = []
for rank, i in enumerate(idx_worst, 1):
    src = val_data[i]['translation']['zh']
    tgt = val_data[i]['translation']['vi']
    pred = preds_ft[i]
    error_rows.append({
        'Rank': rank,
        'BLEU câu': round(sent_bleus[i], 4),
        'Câu Hán nguồn': src,
        'Tham chiếu (Ground Truth)': tgt,
        'Mô hình dự đoán': pred
    })

df_errors = pd.DataFrame(error_rows)
pd.set_option('display.max_colwidth', 200)
print('=== ERROR ANALYSIS: 5 CÂU CÓ BLEU THẤP NHẤT ===')
display(df_errors.style.set_caption('Bảng 8. Phân tích lỗi — 5 câu dịch kém nhất của mô hình fine-tune'))

## Bước 10: Nén kết quả và tải về

Nén toàn bộ kết quả pipeline (TSV, Dataset, Model, Biểu đồ) thành các file zip để tải về.

In [ ]:
# Nén dataset dóng hàng
!zip -q -r alignment_output.zip output/ fig_*.png
print('✅ alignment_output.zip — Kết quả dóng hàng + biểu đồ')

# Nén mô hình đã huấn luyện
!zip -q -r trained_model.zip han_viet_translation_model/
print('✅ trained_model.zip — Mô hình đã fine-tune')

!ls -lh *.zip